# M7 Run 8 - self-supervised pretraining (masked reconstruction)

Pretrains the encoder on folds 1-8, **both datasets, NO WPW label, never fold9/10**. Encoder uses the same module names as the classifier (`stem/layer1/layer2/layer3`) so weights transfer cleanly to Run 9.

**Fully resumable**: step-loop with an **atomic checkpoint every ~200 steps (~5 min)** saving model + optimizer + step + RNG state. Close the computer anytime; on reopen, re-run all cells and Block 8.2 continues exactly where it stopped (at most ~5 min lost). The encoder weights (`encoder.pt`) are re-written at every checkpoint.

> To resume after reopening: just **Run All** again. Block 8.2 detects the checkpoint and picks up.
> Target = 20 epochs (~9 h on CPU); you can stop early, the latest encoder is always on disk.

### Block 8.0: Setup, mmap cache, train/val split, step budget

In [ ]:
# M7 Run 8 - SELF-SUPERVISED PRETRAINING (masked reconstruction) of the encoder on folds 1-8, BOTH datasets.
# NO WPW label, NEVER fold9/fold10. Encoder = ResNet1d conv stack (same module names) -> transfers to Run 9.
# RESUMABLE: step-loop, atomic checkpoint every CKPT_EVERY steps (model+optimizer+step+RNG). Close/reopen safely.
import os, sys, json, time, warnings, shutil, zipfile
import numpy as np
warnings.filterwarnings('ignore')
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
MODELS=os.path.join(ROOT,'models','M7_pretrain'); CACHE_DIR=os.path.join(PROCESSED,'m7_run2_cache')
os.makedirs(MODELS,exist_ok=True); sys.path.insert(0,SRC)
RANDOM_STATE=42; N_JOBS=6
BATCH=64; LR=1e-3; WD=1e-5
TOTAL_EPOCHS=20                 # target passes over folds 1-8 (~836 steps/epoch). Stop anytime; encoder is saved each ckpt.
MASK_RATIO=0.30; MASK_SPAN=100  # mask ~30% of the signal in spans of 100 samples (0.2 s @500 Hz)
CKPT_EVERY=200                  # ~5 min: a crash costs at most this many steps
VAL_EVERY=400; N_VAL=2000       # held-out ECGs (folds 1-8) for reconstruction-loss monitoring only
# reuse Run 2 cache, memory-mapped
NPZ=os.path.join(CACHE_DIR,'m7_run2_data.npz')
def _mmap_member(name):
    out=os.path.join(CACHE_DIR,name+'.npy')
    if not os.path.exists(out):
        with zipfile.ZipFile(NPZ) as zf, zf.open(name+'.npy') as srcf, open(out,'wb') as dst:
            shutil.copyfileobj(srcf,dst,length=16*1024*1024)
    return np.load(out,mmap_mode='r')
X18=_mmap_member('X18'); N=X18.shape[0]; L=X18.shape[2]
rng0=np.random.default_rng(RANDOM_STATE); perm=rng0.permutation(N)
val_idx=np.sort(perm[:N_VAL]); tr_idx=np.sort(perm[N_VAL:])
STEPS_PER_EPOCH=len(tr_idx)//BATCH; TOTAL_STEPS=TOTAL_EPOCHS*STEPS_PER_EPOCH
print('M7 Run 8 pretrain | N %d (train %d / val %d) | L %d | %d steps/epoch | target %d steps'%(
    N,len(tr_idx),len(val_idx),L,STEPS_PER_EPOCH,TOTAL_STEPS))
print('Block 8.0 OK.')


### Block 8.1: Encoder (transferable) + reconstruction decoder

In [ ]:
# Encoder (ResNet1d conv stack, SAME module names as the classifier -> weights transfer) + reconstruction decoder.
import torch, torch.nn as nn, torch.nn.functional as Fnn
torch.set_num_threads(N_JOBS)
class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)
class PretrainNet(nn.Module):
    def __init__(self,ch=(16,32,64)):
        super().__init__()
        # ---- encoder: identical names to ResNet1d (stem, layer1, layer2, layer3) for clean transfer ----
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1); self.layer2=BasicBlock1d(ch[0],ch[1],2); self.layer3=BasicBlock1d(ch[1],ch[2],2)
        # ---- decoder: upsample feature map back to 12 x L (reconstruction) ----
        self.dec=nn.Sequential(
            nn.Conv1d(ch[2],ch[2],3,padding=1), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=4), nn.Conv1d(ch[2],ch[1],5,padding=2), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=4), nn.Conv1d(ch[1],ch[0],5,padding=2), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2), nn.Conv1d(ch[0],12,7,padding=3))
    def encode(self,x): return self.layer3(self.layer2(self.layer1(self.stem(x))))
    def forward(self,x):
        r=self.dec(self.encode(x))
        return Fnn.interpolate(r,size=x.shape[2],mode='linear',align_corners=False)   # exact length
    def encoder_state(self):
        return {k:v for k,v in self.state_dict().items() if k.split('.')[0] in ('stem','layer1','layer2','layer3')}
print('Block 8.1 OK | params:', sum(p.numel() for p in PretrainNet().parameters()))


### Block 8.2: Masking + resumable training loop (atomic checkpoints)

In [ ]:
# Masking + RESUMABLE training loop. Atomic checkpoint (model+opt+step+RNG) every CKPT_EVERY steps.
def get_batch(idx_pool,rng):
    bi=idx_pool[rng.integers(0,len(idx_pool),BATCH)]
    return torch.tensor(np.ascontiguousarray(X18[np.sort(bi)],dtype=np.float32))
def mask_batch(xb,rng):
    B,C,T=xb.shape; m=torch.zeros(B,1,T); nspan=max(1,int(MASK_RATIO*T/MASK_SPAN))
    for b in range(B):
        starts=rng.integers(0,T-MASK_SPAN,nspan)
        for st in starts: m[b,0,st:st+MASK_SPAN]=1.0
    return xb*(1.0-m), (m.expand(B,C,T)>0)
def recon_loss(model,xb,rng):
    xin,mm=mask_batch(xb,rng); r=model(xin)
    return Fnn.mse_loss(r[mm],xb[mm])

model=PretrainNet(); opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
CKPT=os.path.join(MODELS,'pretrain_ckpt.pt'); ENC=os.path.join(MODELS,'encoder.pt')
def save_ckpt(step,best):
    tmp=CKPT+'.tmp'
    torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'step':step,'best':best,
                'torch_rng':torch.get_rng_state(),'np_rng':rng.bit_generator.state},tmp)
    os.replace(tmp,CKPT)                                   # atomic: never a half-written ckpt
    te=ENC+'.tmp'; torch.save(model.encoder_state(),te); os.replace(te,ENC)   # latest encoder always on disk
rng=np.random.default_rng(12345); start_step=0; best=1e9
if os.path.exists(CKPT):
    ck=torch.load(CKPT)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    start_step=ck['step']; best=ck['best']; torch.set_rng_state(ck['torch_rng']); rng.bit_generator.state=ck['np_rng']
    print('[ckpt] resumed at step %d/%d (best val %.5f)'%(start_step,TOTAL_STEPS,best))
else:
    print('[ckpt] fresh start')
t0=time.time(); vrng=np.random.default_rng(7)
for step in range(start_step,TOTAL_STEPS):
    model.train(); xb=get_batch(tr_idx,rng); loss=recon_loss(model,xb,rng)
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
    if step%VAL_EVERY==0:
        model.eval()
        with torch.no_grad():
            vl=np.mean([recon_loss(model,get_batch(val_idx,vrng),vrng).item() for _ in range(8)])
        if vl<best: best=float(vl)
        print('  step %d/%d (ep %.1f) | train %.5f | val %.5f | best %.5f | %.1f min'%(
            step,TOTAL_STEPS,step/STEPS_PER_EPOCH,loss.item(),vl,best,(time.time()-t0)/60))
    if step%CKPT_EVERY==0 and step>start_step:
        save_ckpt(step,best)
save_ckpt(TOTAL_STEPS,best)
print('Block 8.2 OK | pretraining done to step %d | total %.1f min'%(TOTAL_STEPS,(time.time()-t0)/60))


### Block 8.3: Finalize (save encoder for Run 9)

In [ ]:
# Finalize: encoder weights are already saved (encoder.pt) at every checkpoint. Confirm + summary for Run 9.
torch.save(model.encoder_state(),os.path.join(MODELS,'encoder.pt'))
json.dump({'total_steps':TOTAL_STEPS,'epochs':TOTAL_EPOCHS,'best_val_recon':best,
           'mask_ratio':MASK_RATIO,'mask_span':MASK_SPAN,'encoder':'stem+layer1+layer2+layer3'},
          open(os.path.join(MODELS,'pretrain_summary.json'),'w'),indent=2)
print('Block 8.3 OK | encoder saved -> %s'%os.path.join(MODELS,'encoder.pt'))
print('  Run 9 will load these conv weights into ResNet1d (strict=False) and A/B from-scratch vs frozen vs fine-tune.')
